# Correlation Coefficient — Obesity, Diabetes & the Social Determinants of Health
### All 58 California Counties · CDC PLACES Data (2022–2023)

---

## Introduction

Why are some communities sicker than others?

This is one of the central questions in public health and sociology. For decades, the dominant narrative has framed health as a matter of **individual choice** — people get diabetes because they eat poorly, people are obese because they don't exercise. But a growing body of research tells a different story: that where you live, how much money you make, and what resources your community has access to may predict your health outcomes better than any personal behavior.

Sociologists call these the **social determinants of health** — the economic and social conditions that shape the health of individuals and communities. They include income, education, housing stability, food access, transportation, and neighborhood safety. These aren't just background factors; research consistently shows they are the **primary drivers** of health disparities across the United States.

In this notebook, we'll use real CDC data to investigate these relationships ourselves. We'll start with a simple question — is obesity correlated with diabetes across California counties? — and then dig deeper, using the **Pearson correlation coefficient (r)** to measure the strength of relationships between health outcomes and social conditions. By the end, you'll see a pattern emerge in the data that challenges simple explanations and points toward structural inequality as a root cause of poor health.

### What You'll Learn

1. How to read and interpret scatter plots
2. What the Pearson correlation coefficient (r) measures
3. The difference between crude and age-adjusted data
4. How to use a correlation matrix to see patterns across many variables
5. Why correlation does not equal causation — and how to think about confounding variables
6. How social and economic conditions relate to health outcomes

---

## Data Source & Methodology

### The Dataset: CDC PLACES

We are using the **CDC PLACES** dataset (Population Level Analysis and Community EStimates), a program run by the Centers for Disease Control and Prevention. PLACES provides health estimates for every county, city, census tract, and ZIP code in the United States — over 3,000 counties and 30,000+ places.

Our dataset contains **229,298 rows** covering **40 health and social measures** across all 50 states. For our primary analysis, we focus on **California's 58 counties** before expanding to the full national dataset.

### How Were These Numbers Generated?

This is an important question, and one your students should understand: **these are not raw survey counts.** Nobody went door-to-door in each county counting people with diabetes.

Instead, the data was created through a three-step process:

1. **The Survey (BRFSS):** The CDC runs the Behavioral Risk Factor Surveillance System — the largest continuously conducted telephone health survey in the world. Each year, over 400,000 adults across all 50 states are interviewed about their health behaviors, chronic conditions, and access to care. However, BRFSS is designed to be statistically reliable at the **state level**, not the county level.

2. **The Model (MRP):** To get county-level estimates, the CDC uses a statistical technique called **multilevel regression and poststratification (MRP)**. This model takes the state-level survey responses and combines them with county-level demographic data from the U.S. Census Bureau (age, race, income, education) and the American Community Survey. The model essentially asks: "Given what we know about how health varies by demographics at the state level, and given the demographic profile of each county, what would we expect the health rates to be in each county?"

3. **The Estimates:** The output is a model-based estimate for each measure in each county. For example, "Fresno County: 12.4% diabetes prevalence" means the model predicts that approximately 12.4% of adults in Fresno County have diabetes, based on the county's demographics and statewide survey patterns.

### Why This Matters Sociologically

This methodology is important for several reasons:

**It makes the invisible visible.** Before PLACES, most communities had no local health data at all. Public health officials in small counties and rural areas had to rely on state-level averages that obscured enormous local variation. PLACES revealed for the first time that health outcomes can vary dramatically between neighboring counties — and those variations track closely with socioeconomic conditions.

**It reflects structural reality.** The model is built on demographic variables — age, race, income, education — that sociologists recognize as **structural determinants**. The fact that the model works (its estimates closely match direct surveys where both are available) is itself evidence that health outcomes are shaped by social position, not just individual behavior.

**It enables the kind of analysis we're doing.** By providing standardized, comparable health estimates for every county, PLACES allows us to ask questions like: "Is the relationship between poverty and obesity the same in California as it is in Mississippi?" This kind of cross-community comparison is fundamental to sociological research.

### Limitations

- These are **estimates, not exact measurements.** They are the model's best prediction, not a direct count.
- Estimates are more reliable for **larger counties** (more data to work with) than very small ones.
- The model assumes that demographic-health relationships observed at the state level hold at the county level — this is generally true but not always.
- The model **cannot capture effects of local interventions** (e.g., a county that launched a successful diabetes prevention program).
- For our purposes — identifying **patterns and correlations** across many counties — this data is well-suited and widely used in published research.

### Citation

> Greenlund KJ, Lu H, Wang Y, Matthews KA, LeClercq JM, Lee B, et al. *PLACES: Local Data for Better Health.* Prev Chronic Dis 2022;19:210459. DOI: https://doi.org/10.5888/pcd19.210459  
> Data download: https://data.cdc.gov

---

## Step 1: Load the Data

We load the full CDC PLACES dataset and filter it to California counties, looking at just two health measures: **Obesity** and **Diabetes**.

The dataset contains TWO types of prevalence for each county:

| Type | What it means | Analogy |
|------|--------------|--------|
| **Crude (Raw %)** | The actual percentage — no adjustments. If 300 out of 1,000 adults are obese, the crude rate is 30%. | Raw test scores |
| **Age-Adjusted** | What the rate *would be* if every county had the same age mix. Removes the effect of one county having more seniors than another. | Grading on a curve to make it fair |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Load the full CDC dataset
df = pd.read_csv("cdc_places_data.csv", low_memory=False)

# Filter: California only, Diabetes and Obesity only
df_ca = df[
    (df["StateAbbr"] == "CA") &
    (df["MeasureId"].isin(["DIABETES", "OBESITY"]))
].copy()

print(f"California rows for Obesity & Diabetes: {len(df_ca):,}")
print(f"Counties: {df_ca['LocationName'].nunique()}")
print(f"Data Value Types: {df_ca['DataValueTypeID'].unique()}")

## Step 2: Separate Crude and Age-Adjusted Data

We split the data into two tables — one for crude prevalence, one for age-adjusted. Each table has 58 rows (one per county) and two columns (Obesity %, Diabetes %).

In [ ]:
def build_pivot(df_ca, data_type):
    """Pivot one data value type into county x measure format."""
    subset = df_ca[df_ca["DataValueTypeID"] == data_type]
    pivot = subset.pivot_table(
        index="LocationName",
        columns="MeasureId",
        values="Data_Value",
        aggfunc="mean",
    ).dropna()
    return pivot

crude_df = build_pivot(df_ca, "CrdPrv")
age_adj_df = build_pivot(df_ca, "AgeAdjPrv")

print(f"Crude prevalence:        {len(crude_df)} counties")
print(f"Age-adjusted prevalence: {len(age_adj_df)} counties")
print()
print("Sample of CRUDE data (first 5 counties):")
print(crude_df.head())
print()
print("Sample of AGE-ADJUSTED data (first 5 counties):")
print(age_adj_df.head())

## Step 3: Calculate the Correlation Coefficient

The **Pearson correlation coefficient (r)** measures the strength and direction of a linear relationship between two variables.

- **r = 1.0** → perfect positive correlation (as one goes up, the other always goes up)
- **r = 0.0** → no correlation (no relationship)
- **r = -1.0** → perfect negative correlation (as one goes up, the other always goes down)

| r value | Strength |
|---------|----------|
| 0.8 – 1.0 | Very Strong |
| 0.6 – 0.8 | Strong |
| 0.4 – 0.6 | Moderate |
| 0.2 – 0.4 | Weak |
| 0.0 – 0.2 | Very Weak / None |

In [ ]:
# Calculate Pearson r for both datasets
r_crude, p_crude = stats.pearsonr(crude_df["OBESITY"], crude_df["DIABETES"])
r_adj, p_adj = stats.pearsonr(age_adj_df["OBESITY"], age_adj_df["DIABETES"])

print("CORRELATION COEFFICIENTS")
print("=" * 40)
print(f"  Crude (Raw %):   r = {r_crude:.4f}")
print(f"  Age-Adjusted:    r = {r_adj:.4f}")
print("=" * 40)

## Step 4: Interactive Scatter Plot — Toggle Between Views

Use the dropdown below to switch between **Crude** and **Age-Adjusted** data.

Watch what happens:
- The dots shift position (some counties move more than others)
- The correlation coefficient (r) changes
- The description below the chart explains what you're seeing

In [ ]:
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
except ImportError:
    !pip install ipywidgets -q
    import ipywidgets as widgets


def plot_scatter(view_type):
    """Draw scatter plot for the selected data type."""

    if view_type == "Crude (Raw %)":
        data = crude_df
        r_val = r_crude
        dot_color = "#E07A3A"
        description = (
            "CRUDE (RAW %) — What you see is what you get\n\n"
            "This is the actual percentage of adults with each condition. "
            "If 300 out of 1,000 adults in Fresno County are obese, "
            "the crude rate is 30%. No adjustments — just the raw number.\n\n"
            "Keep in mind: Some counties have older populations, which "
            "naturally increases obesity and diabetes rates. So part of what "
            "you're seeing might just be age differences between counties, "
            "not true health differences."
        )
    else:
        data = age_adj_df
        r_val = r_adj
        dot_color = "#2E6B8A"
        description = (
            "AGE-ADJUSTED — Apples to apples comparison\n\n"
            "This answers: 'What would each county's rate be if they all "
            "had the same age mix?' The CDC statistically removes the "
            "effect of having an older or younger population.\n\n"
            "Now we're comparing fairly. If a county STILL shows high "
            "obesity after age-adjustment, it's not just because of older "
            "residents — something else is going on (diet, income, access "
            "to healthcare, physical activity, etc.)."
        )

    # Classify strength
    a = abs(r_val)
    if a >= 0.8:
        strength = "Very Strong"
    elif a >= 0.6:
        strength = "Strong"
    elif a >= 0.4:
        strength = "Moderate"
    elif a >= 0.2:
        strength = "Weak"
    else:
        strength = "Very Weak / None"

    # ── Create the figure ──
    fig, ax = plt.subplots(figsize=(12, 8))

    # Scatter dots — OBESITY on X, DIABETES on Y
    ax.scatter(
        data["OBESITY"], data["DIABETES"],
        s=80, alpha=0.7, color=dot_color,
        edgecolors="white", linewidths=0.8, zorder=3,
    )

    # Label every county
    for county, row in data.iterrows():
        ax.annotate(
            county, (row["OBESITY"], row["DIABETES"]),
            fontsize=7, alpha=0.7, color="#555",
            xytext=(5, 4), textcoords="offset points",
        )

    # Titles and labels
    ax.set_title(
        f"Obesity vs Diabetes — 58 California Counties\n"
        f"Correlation Coefficient:  r = {r_val:.4f}  ({strength} Positive Correlation)",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Obesity Prevalence — % of Adults", fontsize=12, labelpad=10)
    ax.set_ylabel("Diabetes Prevalence — % of Adults", fontsize=12, labelpad=10)

    # Grid
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_xlim(14, 40)
    ax.set_ylim(6.5, 16.5)

    # Stats box
    stats_text = (
        f"n = {len(data)} counties\n"
        f"r = {r_val:.4f}"
    )
    ax.text(
        0.98, 0.04, stats_text,
        transform=ax.transAxes, fontsize=11,
        fontfamily="monospace", verticalalignment="bottom",
        horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                  edgecolor="#ccc", alpha=0.9),
    )

    # Source
    fig.text(
        0.5, -0.02,
        "Data: CDC PLACES — Local Data for Better Health (2022–2023)  ·  "
        "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459  ·  data.cdc.gov",
        ha="center", fontsize=8, color="#aaa", style="italic",
    )

    plt.tight_layout()
    plt.show()

    # Print description below the chart
    print("─" * 60)
    print(description)
    print("─" * 60)


# ── Create the toggle dropdown ──
toggle = widgets.Dropdown(
    options=["Crude (Raw %)", "Age-Adjusted"],
    value="Crude (Raw %)",
    description="View:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="250px"),
)

output = widgets.interactive_output(plot_scatter, {"view_type": toggle})

display(toggle, output)

## Step 5: Compare the Two Views Side by Side

Here we see both scatter plots at the same time so we can directly compare how the dots shift when we remove the age effect.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

for ax, data, r_val, title, color in [
    (ax1, crude_df, r_crude, "Crude (Raw %)", "#E07A3A"),
    (ax2, age_adj_df, r_adj, "Age-Adjusted", "#2E6B8A"),
]:
    # OBESITY on X, DIABETES on Y
    ax.scatter(
        data["OBESITY"], data["DIABETES"],
        s=70, alpha=0.7, color=color,
        edgecolors="white", linewidths=0.8, zorder=3,
    )

    for county, row in data.iterrows():
        ax.annotate(
            county, (row["OBESITY"], row["DIABETES"]),
            fontsize=6, alpha=0.6, color="#555",
            xytext=(4, 3), textcoords="offset points",
        )

    ax.set_title(f"{title}\nr = {r_val:.4f}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Obesity Prevalence — % of Adults", fontsize=10)
    ax.set_ylabel("Diabetes Prevalence — % of Adults", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_xlim(14, 40)
    ax.set_ylim(6.5, 16.5)

fig.suptitle(
    "Obesity vs Diabetes — 58 California Counties\n"
    "How does the correlation coefficient change when we adjust for age?",
    fontsize=15, fontweight="bold", y=1.02,
)

fig.text(
    0.5, -0.02,
    "Data: CDC PLACES (2022–2023)  ·  Greenlund KJ et al., Prev Chronic Dis 2022;19:210459",
    ha="center", fontsize=8, color="#aaa", style="italic",
)

plt.tight_layout()
plt.show()

print()
print("WHAT CHANGED?")
print("=" * 50)
print(f"  Crude r:        {r_crude:.4f}")
print(f"  Age-Adjusted r: {r_adj:.4f}")
print(f"  Difference:     {r_adj - r_crude:+.4f}")
print()
print("The age-adjusted r is slightly HIGHER. Why?")
print("Because age was adding 'noise' to the relationship.")
print("Some counties looked like they had high obesity")
print("and high diabetes just because they had more seniors.")
print("Once we remove that age effect, the underlying")
print("connection between obesity and diabetes becomes")
print("a little clearer.")

## Discussion Questions

1. **What does r = 0.63 (or 0.67) tell us?**  
   It tells us there's a strong positive association — counties with higher obesity rates tend to also have higher diabetes rates.

2. **Does this mean obesity CAUSES diabetes?**  
   Not from this data alone! Correlation tells us two things move together — it does NOT tell us that one causes the other. We placed obesity on the X-axis because research suggests it's a risk factor for diabetes, but this scatter plot alone cannot prove causation.

3. **What else might cause BOTH obesity and diabetes to be high in the same counties?**  
   Think about: poverty, food deserts (limited access to healthy food), less access to healthcare, fewer parks and places to exercise, cultural factors, education levels.

4. **Why did r go UP when we age-adjusted?**  
   Because age was a *confounding variable*. Older people are more likely to have both conditions. Some counties appeared to have high rates simply because they had older populations. Removing that effect revealed a slightly stronger underlying relationship.

5. **Can you find your county on the chart? Where does it fall?**

---

*Data: CDC PLACES — Local Data for Better Health, County Data (2022–2023 Release)*  
*Citation: Greenlund KJ, Lu H, Wang Y, et al. PLACES: Local Data for Better Health. Prev Chronic Dis 2022;19:210459.*

## Step 6: Explore — What Else Is Connected to Obesity?

We just saw that obesity and diabetes are strongly correlated (r = 0.67). But we also asked: **does obesity cause diabetes, or is something else driving both?**

The CDC dataset has 40 health measures for every California county. Let's explore some of them. Use the dropdown below to pick a measure and see how it correlates with obesity.

As you click through, ask yourself:
- Which measures have the **strongest** correlation with obesity?
- Do you notice a **pattern** in what's strongly correlated?
- What does that pattern tell us about **why** some counties have more obesity than others?
- Are there any **surprises** — things you expected to correlate that don't?

In [ ]:
# ── Build the full age-adjusted pivot table with all measures ──

adj_all = df[df["DataValueTypeID"] == "AgeAdjPrv"]
adj_ca = adj_all[adj_all["StateAbbr"] == "CA"]

pivot_all = adj_ca.pivot_table(
    index="LocationName",
    columns="MeasureId",
    values="Data_Value",
    aggfunc="mean",
).dropna(axis=1, how="all")

print(f"Full pivot table: {pivot_all.shape[0]} counties × {pivot_all.shape[1]} measures")
print(f"Measures available: {sorted(pivot_all.columns.tolist())}")
# Curated measures with in-depth descriptions

EXPLORE_MEASURES = {
    "DIABETES": {
        "label": "Diabetes",
        "group": "Already Studied",
        "y_label": "Diabetes Prevalence — % of Adults",
        "description": (
            "You already studied this one — counties with more obesity tend to have more "
            "diabetes (r = 0.67). But here's the question that drives all of social science: "
            "is this a direct cause-and-effect relationship, or are we seeing two symptoms "
            "of a deeper problem? As you explore the other measures below, keep asking "
            "yourself: do the same counties keep appearing in the same spots?"
        ),
    },
    "LPA": {
        "label": "Physical Inactivity",
        "group": "Already Studied",
        "y_label": "Physical Inactivity — % of Adults",
        "description": (
            "Counties where fewer adults exercise have higher obesity rates. This seems "
            "straightforward — less movement, more weight gain. But sociologists ask a "
            "deeper question: WHY are people in these counties less active? Research shows "
            "that low-income communities often lack sidewalks, parks, and safe spaces to "
            "exercise. When your neighborhood doesn't have a single park and the streets "
            "aren't safe to walk at night, 'choosing to exercise' isn't really a choice. "
            "Physical inactivity may be as much about zip code as willpower."
        ),
    },
    "FOODSTAMP": {
        "label": "Food Stamps",
        "group": "Social & Economic Factors",
        "y_label": "Food Stamps — % of Adults Receiving Assistance",
        "description": (
            "This is the percentage of adults receiving government food assistance — a direct "
            "measure of poverty. The fact that this correlates so strongly with obesity "
            "challenges a common misconception: that obesity is simply about eating too much. "
            "In reality, the cheapest and most accessible foods in low-income areas tend to "
            "be highly processed, calorie-dense, and nutrient-poor. A bag of chips costs $1. "
            "A bag of fresh vegetables costs $5. When your budget is tight, the 'unhealthy "
            "choice' is often the only affordable one. Sociologists call this a structural "
            "constraint — the environment limits your options regardless of your personal "
            "preferences."
        ),
    },
    "FOODINSECU": {
        "label": "Food Insecurity",
        "group": "Social & Economic Factors",
        "y_label": "Food Insecurity — % of Adults",
        "description": (
            "Food insecurity means not knowing where your next meal will come from. "
            "Paradoxically, food insecurity is strongly linked to obesity — not starvation. "
            "Why? When food is uncertain, people tend to eat large quantities of cheap, "
            "calorie-dense food when it IS available, because they don't know when they'll "
            "eat again. This cycle of scarcity and overeating is well-documented in public "
            "health research. It's a powerful example of how poverty changes behavior in "
            "ways that outsiders might not expect."
        ),
    },
    "HOUSINSECU": {
        "label": "Housing Insecurity",
        "group": "Social & Economic Factors",
        "y_label": "Housing Insecurity — % of Adults",
        "description": (
            "People worried about paying rent or losing their home. What does housing have "
            "to do with obesity? More than you'd think. Chronic financial stress triggers "
            "cortisol, a hormone that promotes fat storage — especially around the abdomen. "
            "When you're spending 60% of your income on rent, you're not buying fresh "
            "groceries or a gym membership. You're buying the cheapest calories you can find "
            "and working multiple jobs that leave no time for cooking or exercise. Housing "
            "insecurity is a window into the daily reality of poverty — and its health "
            "consequences."
        ),
    },
    "LACKTRPT": {
        "label": "Transportation Barriers",
        "group": "Social & Economic Factors",
        "y_label": "Transportation Barriers — % of Adults",
        "description": (
            "People who can't reliably get where they need to go. No car and no bus route "
            "might mean: no access to a full grocery store (only the corner store with "
            "processed food), no way to get to a gym or a park, no way to get to a doctor "
            "for preventive care, and longer commutes that eat into time for cooking and "
            "exercise. In rural California counties, the nearest supermarket can be 30+ "
            "miles away. Transportation isn't just about convenience — it determines what "
            "you can eat, where you can move, and whether you can see a doctor."
        ),
    },
    "BPHIGH": {
        "label": "High Blood Pressure",
        "group": "Health Consequences",
        "y_label": "High Blood Pressure — % of Adults",
        "description": (
            "High blood pressure (hypertension) is one of the strongest correlates with "
            "obesity in this dataset. Medically, this makes sense — excess weight puts "
            "strain on the cardiovascular system. But notice something sociological: the "
            "counties with the highest blood pressure are the same ones showing up at the "
            "top of the food stamps, housing insecurity, and transportation charts. This is "
            "what researchers call a 'clustering of disadvantage' — poor communities don't "
            "just face one health problem. They face many, all at once, all reinforcing "
            "each other. This is why public health experts argue that you can't fix obesity "
            "without fixing poverty."
        ),
    },
    "CHD": {
        "label": "Coronary Heart Disease",
        "group": "Health Consequences",
        "y_label": "Coronary Heart Disease — % of Adults",
        "description": (
            "Heart disease follows the exact same geographic pattern as obesity, diabetes, "
            "high blood pressure, and poverty. This is not a coincidence. In sociology, we "
            "call this the 'social determinants of health' — the idea that your health "
            "outcomes are shaped less by your individual choices and more by the conditions "
            "you were born into. The county you live in, your family's income, your access "
            "to food and healthcare — these structural factors predict heart disease better "
            "than any individual behavior. When you see the same counties clustered at the "
            "high end of every chart, you're seeing structural inequality made visible "
            "through data."
        ),
    },
    "BINGE": {
        "label": "Binge Drinking",
        "group": "Surprises",
        "y_label": "Binge Drinking — % of Adults",
        "description": (
            "Here's your first surprise — almost no correlation. You might have assumed "
            "that binge drinking would track with obesity, but it doesn't. The dots are "
            "scattered with no clear pattern. Why? Binge drinking cuts across income levels "
            "in ways that obesity doesn't. Wealthier counties actually tend to have HIGHER "
            "binge drinking rates. This is important methodologically: it shows that not "
            "every health behavior follows the same socioeconomic pattern. It also proves "
            "that the strong correlations we found earlier aren't just an artifact of the "
            "data — they reflect real, specific relationships."
        ),
    },
    "DENTAL": {
        "label": "Dental Visits",
        "group": "Surprises",
        "y_label": "Dental Visits — % of Adults",
        "description": (
            "This is your first negative correlation — the line goes DOWN. Counties with "
            "more obesity have FEWER dental visits. Dental care is one of the most "
            "income-sensitive health services in America: it's expensive, often not covered "
            "by insurance, and frequently the first thing people cut when money is tight. "
            "This negative correlation is another poverty signal. The same counties where "
            "people can't afford dentists are the counties where obesity, diabetes, and "
            "heart disease are highest. It's all connected — and it's all connected "
            "through economics."
        ),
    },
}

print(f"Curated measures for exploration: {len(EXPLORE_MEASURES)}")
print()
current_group = ""
for key, info in EXPLORE_MEASURES.items():
    if info["group"] != current_group:
        current_group = info["group"]
        print(f"  -- {current_group} --")
    label = info["label"]
    print(f"     {label:<28} ({key})")

# Interactive Explorer with grouped dropdown

def r_strength(r):
    a = abs(r)
    if a >= 0.8: return "Very Strong"
    elif a >= 0.6: return "Strong"
    elif a >= 0.4: return "Moderate"
    elif a >= 0.2: return "Weak"
    else: return "Very Weak / None"


def explore_measure(measure_name):
    """Draw scatter plot of Obesity vs selected measure."""

    # Skip if separator was selected
    if measure_name.startswith("──"):
        print("Please select a measure from the dropdown.")
        return

    # Find the measure key from the label
    measure_key = None
    for key, info in EXPLORE_MEASURES.items():
        if info["label"] == measure_name:
            measure_key = key
            break

    if measure_key is None:
        print("Please select a measure from the dropdown.")
        return

    info = EXPLORE_MEASURES[measure_key]
    clean = pivot_all[["OBESITY", measure_key]].dropna()
    r_val, p_val = stats.pearsonr(clean["OBESITY"], clean[measure_key])

    # Determine direction and color
    if r_val > 0.2:
        direction = "Positive"
        dot_color = "#E07A3A"
    elif r_val < -0.2:
        direction = "Negative"
        dot_color = "#2E6B8A"
    else:
        direction = "No"
        dot_color = "#888888"

    strength = r_strength(r_val)

    # Create the figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # Scatter — OBESITY always on X
    ax.scatter(
        clean["OBESITY"], clean[measure_key],
        s=80, alpha=0.7, color=dot_color,
        edgecolors="white", linewidths=0.8, zorder=3,
    )

    # Label every county
    for county in clean.index:
        ax.annotate(
            county,
            (clean.loc[county, "OBESITY"], clean.loc[county, measure_key]),
            fontsize=7, alpha=0.7, color="#555",
            xytext=(5, 4), textcoords="offset points",
        )

    # Title with r value
    title_line1 = "Obesity vs " + info["label"] + " — 58 California Counties"
    title_line2 = "Correlation Coefficient:  r = " + f"{r_val:.4f}" + "  (" + strength + " " + direction + " Correlation)"
    ax.set_title(title_line1 + chr(10) + title_line2, fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Obesity Prevalence — % of Adults", fontsize=12, labelpad=10)
    ax.set_ylabel(info["y_label"], fontsize=12, labelpad=10)

    # Grid
    ax.grid(True, linestyle="--", alpha=0.3)

    # Stats box
    stats_line = "n = " + str(len(clean)) + " counties" + chr(10) + "r = " + f"{r_val:.4f}"
    ax.text(
        0.98, 0.04, stats_line,
        transform=ax.transAxes, fontsize=11,
        fontfamily="monospace", verticalalignment="bottom",
        horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                  edgecolor="#ccc", alpha=0.9),
    )

    # Source
    fig.text(
        0.5, -0.02,
        "Data: CDC PLACES — Local Data for Better Health (2022-2023)  |  "
        "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459  |  data.cdc.gov",
        ha="center", fontsize=8, color="#aaa", style="italic",
    )

    plt.tight_layout()
    plt.show()

    # Print the description
    print("=" * 60)
    print(info["label"].upper() + " vs OBESITY")
    print("r = " + f"{r_val:.4f}" + "  |  " + strength + " " + direction + " Correlation")
    print("=" * 60)
    print()
    print(info["description"])
    print()
    print("=" * 60)


# Build grouped dropdown options
dropdown_options = []
current_group = ""
for key, info in EXPLORE_MEASURES.items():
    if info["group"] != current_group:
        current_group = info["group"]
        separator = "── " + current_group + " ──"
        dropdown_options.append(separator)
    dropdown_options.append(info["label"])

explore_toggle = widgets.Dropdown(
    options=dropdown_options,
    value="Diabetes",
    description="Measure:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="400px"),
)

explore_output = widgets.interactive_output(explore_measure, {"measure_name": explore_toggle})

display(explore_toggle, explore_output)


## Step 7: The Big Picture — Correlation Matrix Heatmap

### What is a Correlation Coefficient Matrix?

In Step 6, the dropdown let you explore one relationship at a time — obesity vs food stamps, then obesity vs diabetes, then obesity vs dental visits. That's useful, but it only shows relationships **with obesity**. What about the relationship between food stamps and diabetes? Or between transportation barriers and heart disease?

A **correlation coefficient matrix** answers all of those questions simultaneously. It's a table that shows the Pearson r value between **every possible pair** of variables in your dataset, all at once. If you have 11 measures, you get an 11×11 grid — that's 55 unique pairs of relationships visible in one table. Instead of clicking through a dropdown 55 times, you see everything at a glance.

### Why Does This Matter?

When you looked at individual scatter plots, you might have thought "ok, obesity correlates with food stamps, that's interesting." But the matrix reveals something bigger: food stamps ALSO correlates with diabetes, AND with heart disease, AND with housing insecurity, AND with transportation barriers — all at r values above 0.75. It's not a bunch of separate relationships. It's a **web**. Everything in the poverty cluster is connected to everything else.

That's the difference between seeing one thread and seeing the whole web. The matrix reveals **structure** in the data that individual scatter plots can't show you.

### Why Researchers Use This

In social science research, the correlation matrix is often the first thing a researcher looks at before doing any deeper analysis. It tells you which variables move together, which ones are independent, and where to look for confounding variables. It's the map before the journey.

### How to Read the Heatmap Below

- **Dark red** = strong positive correlation (both go up together)
- **Dark blue** = strong negative correlation (one goes up, the other goes down)
- **White** = no correlation (no relationship)
- The **diagonal** is always 1.00 (every measure correlates perfectly with itself)

Look for the **cluster** — the block of dark red that connects poverty indicators, obesity, and health outcomes. That cluster is the story of this entire lesson.

In [ ]:
# Correlation Heatmap — all 10 curated measures

import matplotlib.colors as mcolors

# Build a dataframe with just our 10 measures, using readable names
measure_rename = {
    "OBESITY": "Obesity",
    "DIABETES": "Diabetes",
    "LPA": "Physical Inactivity",
    "FOODSTAMP": "Food Stamps",
    "FOODINSECU": "Food Insecurity",
    "HOUSINSECU": "Housing Insecurity",
    "LACKTRPT": "Transportation Barriers",
    "BPHIGH": "High Blood Pressure",
    "CHD": "Heart Disease",
    "BINGE": "Binge Drinking",
    "DENTAL": "Dental Visits",
}

# Order them so the poverty cluster appears together
measure_order = [
    "OBESITY", "DIABETES", "BPHIGH", "CHD",
    "LPA", "FOODSTAMP", "FOODINSECU", "HOUSINSECU", "LACKTRPT",
    "DENTAL", "BINGE",
]

heatmap_df = pivot_all[measure_order].rename(columns=measure_rename)
corr_matrix = heatmap_df.corr()

# Create the heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Custom colormap: blue (negative) -> white (zero) -> red (positive)
cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom", ["#2E6B8A", "#5A9BB5", "#FFFFFF", "#E8956A", "#E07A3A"]
)

# Draw the heatmap
im = ax.imshow(corr_matrix.values, cmap=cmap, vmin=-1, vmax=1, aspect="equal")

# Labels
labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(labels, fontsize=10)

# Add r values in each cell
for i in range(len(labels)):
    for j in range(len(labels)):
        r_val = corr_matrix.values[i, j]
        # White text on dark cells, black on light cells
        text_color = "white" if abs(r_val) > 0.65 else "black"
        ax.text(j, i, f"{r_val:.2f}",
                ha="center", va="center", fontsize=9,
                fontweight="bold" if abs(r_val) >= 0.8 else "normal",
                color=text_color)

# Colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label("Pearson Correlation Coefficient (r)", fontsize=11)

# Title
ax.set_title(
    "Correlation Heatmap — 11 Health & Social Measures" + chr(10) +
    "58 California Counties (Age-Adjusted, CDC PLACES 2022-2023)",
    fontsize=14, fontweight="bold", pad=20,
)

# Source
fig.text(
    0.5, -0.01,
    "Data: CDC PLACES — Local Data for Better Health (2022-2023)  |  "
    "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459  |  data.cdc.gov",
    ha="center", fontsize=8, color="#aaa", style="italic",
)

plt.tight_layout()
plt.show()

print()
print("=" * 60)
print("  READING THE HEATMAP")
print("=" * 60)
print()
print("Each cell shows the r value between two measures.")
print("The diagonal is always 1.00 (every measure correlates")
print("perfectly with itself).")
print()
print("WHAT TO NOTICE:")
print()
print("1. THE RED CLUSTER (top-left and center):")
print("   Obesity, Diabetes, High Blood Pressure, Heart Disease,")
print("   Food Stamps, Food Insecurity, Housing Insecurity, and")
print("   Transportation Barriers are ALL strongly correlated")
print("   with EACH OTHER. This isn't 10 separate relationships —")
print("   it's one interconnected web of poverty and poor health.")
print()
print("2. THE BLUE COLUMN (Dental Visits):")
print("   Dental Visits is negatively correlated with almost")
print("   everything in the red cluster. Less dental care = more")
print("   poverty = more obesity = more disease. Same story,")
print("   opposite direction.")
print()
print("3. THE WHITE ROW (Binge Drinking):")
print("   Binge Drinking is disconnected from the cluster.")
print("   It doesn't follow the poverty pattern. This confirms")
print("   that the red cluster isn't just 'everything correlates")
print("   with everything' — it's a specific pattern tied to")
print("   socioeconomic conditions.")
print()
print("=" * 60)


## Step 8: What Did We Discover?

After exploring the measures above, a pattern emerges:

**The strongest correlations with obesity aren't just other diseases — they're signs of poverty.**

| Measure | r | What it tells us |
|---------|---|------------------|
| High Blood Pressure | +0.89 | Health consequence — clusters with obesity |
| Coronary Heart Disease | +0.86 | Health consequence — same counties, same problems |
| Transportation Barriers | +0.83 | Poverty signal — can't get to stores, gyms, doctors |
| Food Stamps | +0.83 | Poverty signal — low income |
| Housing Insecurity | +0.79 | Poverty signal — can't afford stable housing |
| Physical Inactivity | +0.77 | Behavior — but driven by environment? |
| Food Insecurity | +0.77 | Poverty signal — not enough food access |
| Dental Visits | -0.77 | Poverty signal (negative) — can't afford dental care |
| Diabetes | +0.67 | Health consequence — where we started |
| Binge Drinking | -0.14 | No correlation — not everything is connected |

### The Big Takeaway

When we started, we saw that obesity and diabetes were correlated and asked: **does obesity cause diabetes?**

Now we can see that the same counties with high obesity and high diabetes also have high food insecurity, high housing insecurity, more people on food stamps, and fewer dental visits. These are all markers of **poverty**.

This suggests that **poverty may be the confounding variable** — it doesn't just cause obesity OR diabetes. It creates conditions (cheap processed food, no safe places to exercise, limited healthcare access, chronic stress) that lead to BOTH.

This is why **correlation does not equal causation**. The scatter plot showed us that obesity and diabetes move together, but it took deeper exploration to see that a third factor — economic hardship — may be driving both.

---

*Data: CDC PLACES — Local Data for Better Health, County Data (2022–2023 Release)*  
*Citation: Greenlund KJ, Lu H, Wang Y, et al. PLACES: Local Data for Better Health. Prev Chronic Dis 2022;19:210459.*

## Step 9: Does This Pattern Hold in Other States?

Everything we've found so far is based on California's 58 counties. But is the poverty-health cluster a California-specific phenomenon, or does it show up everywhere?

The full CDC PLACES dataset covers **all 50 states**. Use the dropdown below to pick any state and generate its correlation heatmap. If the same red cluster appears in Mississippi, Massachusetts, and Montana — states with very different populations, politics, and cultures — that's strong evidence that we're looking at a **structural pattern**, not a local anomaly.

Some things to watch for:
- Does the poverty cluster (food stamps, food insecurity, housing insecurity, transportation) still show up as a block of red?
- Is binge drinking still disconnected from the cluster?
- Are the r values stronger or weaker in different states?
- Some states may be missing social needs measures (food stamps, housing insecurity, etc.) — the heatmap will adjust automatically.

In [ ]:
# Step 9: State-level correlation heatmap explorer

import matplotlib.colors as mcolors

# Get full state names for the dropdown
state_names = df[["StateAbbr", "StateDesc"]].drop_duplicates().sort_values("StateDesc")
state_names = state_names[state_names["StateAbbr"] != "US"]  # Remove national aggregate

# Build lookup
state_abbr_to_name = dict(zip(state_names["StateAbbr"], state_names["StateDesc"]))
state_name_to_abbr = dict(zip(state_names["StateDesc"], state_names["StateAbbr"]))

# Our 11 measures in display order
measure_order = [
    "OBESITY", "DIABETES", "BPHIGH", "CHD",
    "LPA", "FOODSTAMP", "FOODINSECU", "HOUSINSECU", "LACKTRPT",
    "DENTAL", "BINGE",
]
measure_rename = {
    "OBESITY": "Obesity", "DIABETES": "Diabetes",
    "BPHIGH": "High Blood Pressure", "CHD": "Heart Disease",
    "LPA": "Physical Inactivity", "FOODSTAMP": "Food Stamps",
    "FOODINSECU": "Food Insecurity", "HOUSINSECU": "Housing Insecurity",
    "LACKTRPT": "Transportation Barriers", "DENTAL": "Dental Visits",
    "BINGE": "Binge Drinking",
}

# Custom colormap
heatmap_cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom", ["#2E6B8A", "#5A9BB5", "#FFFFFF", "#E8956A", "#E07A3A"]
)


def sample_size_warning(n_counties):
    """Return a warning string based on county count."""
    if n_counties < 5:
        return "NOT RELIABLE"
    elif n_counties < 15:
        return "USE EXTREME CAUTION"
    elif n_counties < 30:
        return "INTERPRET WITH CAUTION"
    else:
        return "RELIABLE"


def build_state_heatmap(state_name, ax=None, figsize=(12, 10), show_source=True):
    """Build correlation heatmap for a given state. Returns the corr matrix."""

    abbr = state_name_to_abbr[state_name]
    adj_state = df[(df["StateAbbr"] == abbr) & (df["DataValueTypeID"] == "AgeAdjPrv")]

    pivot = adj_state.pivot_table(
        index="LocationName", columns="MeasureId",
        values="Data_Value", aggfunc="mean",
    )

    # Use only measures that exist in this state
    available = [m for m in measure_order if m in pivot.columns]
    rename_available = {m: measure_rename[m] for m in available}
    heatmap_df = pivot[available].rename(columns=rename_available).dropna()
    n_counties = len(heatmap_df)

    if n_counties < 3:
        print("=" * 50)
        print("  NOT ENOUGH DATA")
        print("=" * 50)
        print()
        print("  " + state_name + " has only " + str(n_counties) + " counties in this dataset.")
        print("  You need at least 3 data points to calculate a correlation,")
        print("  and many more for it to be meaningful.")
        print("  Try a state with more counties.")
        return None

    corr = heatmap_df.corr()
    reliability = sample_size_warning(n_counties)

    # Create figure if no axis provided
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(corr.values, cmap=heatmap_cmap, vmin=-1, vmax=1, aspect="equal")

    labels = corr.columns.tolist()
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    for i in range(len(labels)):
        for j in range(len(labels)):
            r_val = corr.values[i, j]
            text_color = "white" if abs(r_val) > 0.65 else "black"
            ax.text(j, i, f"{r_val:.2f}",
                    ha="center", va="center", fontsize=8,
                    fontweight="bold" if abs(r_val) >= 0.8 else "normal",
                    color=text_color)

    missing = [measure_rename[m] for m in measure_order if m not in available]
    missing_note = ""
    if missing:
        missing_note = " | missing: " + ", ".join(missing)

    # Title includes county count and reliability
    title_text = state_name + " — " + str(n_counties) + " counties"
    if reliability != "RELIABLE":
        title_text = title_text + " (" + reliability + ")"
    if missing_note:
        title_text = title_text + chr(10) + missing_note.strip()

    ax.set_title(title_text, fontsize=12, fontweight="bold", pad=15)

    if standalone:
        cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label("Pearson r", fontsize=11)
        if show_source:
            fig.text(
                0.5, -0.01,
                "Data: CDC PLACES (2022-2023)  |  "
                "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459",
                ha="center", fontsize=8, color="#aaa", style="italic",
            )
        plt.tight_layout()
        plt.show()

        # Print summary with reliability warning
        print()
        print("=" * 55)
        print("  " + state_name.upper())
        print("  " + str(n_counties) + " counties  |  Age-Adjusted data")
        print("=" * 55)
        print()

        # Sample size guidance
        if n_counties >= 30:
            print("  Sample size: " + str(n_counties) + " counties — RELIABLE")
            print("  This is a large enough sample for meaningful correlations.")
        elif n_counties >= 15:
            print("  Sample size: " + str(n_counties) + " counties — INTERPRET WITH CAUTION")
            print("  This is a moderate sample. The general patterns are likely")
            print("  real, but individual r values could shift significantly")
            print("  with a few more or fewer counties.")
        elif n_counties >= 5:
            print("  Sample size: " + str(n_counties) + " counties — USE EXTREME CAUTION")
            print("  This sample is very small. Correlations based on fewer")
            print("  than 15 data points are unstable — a single outlier")
            print("  county can dramatically change the r value. Do not draw")
            print("  strong conclusions from this heatmap.")
        else:
            print("  Sample size: " + str(n_counties) + " counties — NOT RELIABLE")
            print("  With fewer than 5 data points, correlation is essentially")
            print("  meaningless. Try a state with more counties.")

        if missing:
            print()
            print("  Missing measures: " + ", ".join(missing))
            print("  (These measures are not available for this state in the dataset)")

        print()
        print("=" * 55)

    return corr


def show_state_heatmap(state_name):
    """Wrapper for the dropdown."""
    build_state_heatmap(state_name)


# Build dropdown with state names
state_list = sorted(state_name_to_abbr.keys())

state_dropdown = widgets.Dropdown(
    options=state_list,
    value="California",
    description="State:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="300px"),
)

state_output = widgets.interactive_output(show_state_heatmap, {"state_name": state_dropdown})

display(state_dropdown, state_output)


## Step 10: Compare Two States Side by Side

Now let's put two states next to each other. Pick two states with different demographics, economies, or geographies and compare their heatmaps.

Some interesting comparisons to try:
- **California vs Mississippi** — wealthy coastal state vs one of the poorest states in the nation
- **Massachusetts vs West Virginia** — two very different economic realities
- **New York vs Alabama** — urban-heavy vs rural-heavy

If the same poverty-health cluster appears in both states despite their differences, that's powerful evidence that this pattern is **structural** — built into how poverty affects health everywhere, not just in one place.

In [ ]:
# Step 10: Side-by-side state comparison

def compare_states(state_1, state_2):
    """Draw two heatmaps side by side."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))

    corr1 = build_state_heatmap(state_1, ax=ax1, show_source=False)
    corr2 = build_state_heatmap(state_2, ax=ax2, show_source=False)

    # Add a shared colorbar
    norm = mcolors.Normalize(vmin=-1, vmax=1)
    sm = plt.cm.ScalarMappable(cmap=heatmap_cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax1, ax2], shrink=0.8, pad=0.02)
    cbar.set_label("Pearson r", fontsize=11)

    fig.suptitle(
        "Correlation Heatmap Comparison" + chr(10) +
        "Do the same patterns appear in different states?",
        fontsize=15, fontweight="bold", y=1.02,
    )

    fig.text(
        0.5, -0.01,
        "Data: CDC PLACES (2022-2023)  |  "
        "Greenlund KJ et al., Prev Chronic Dis 2022;19:210459",
        ha="center", fontsize=8, color="#aaa", style="italic",
    )

    plt.tight_layout()
    plt.show()

    # Compare key correlations
    if corr1 is not None and corr2 is not None:
        print()
        print("=" * 60)
        print("  KEY COMPARISONS")
        print("=" * 60)

        pairs = [
            ("Obesity", "Diabetes"),
            ("Obesity", "High Blood Pressure"),
            ("Obesity", "Heart Disease"),
            ("Obesity", "Food Stamps"),
            ("Obesity", "Food Insecurity"),
            ("Obesity", "Dental Visits"),
            ("Obesity", "Binge Drinking"),
        ]

        header = f"{'Measure Pair':<40} {state_1:<14} {state_2:<14}"
        print("  " + header)
        print("  " + "-" * 68)

        for m1, m2 in pairs:
            r1_str = "N/A"
            r2_str = "N/A"
            if m1 in corr1.columns and m2 in corr1.columns:
                r1_str = f"{corr1.loc[m1, m2]:.4f}"
            if m1 in corr2.columns and m2 in corr2.columns:
                r2_str = f"{corr2.loc[m1, m2]:.4f}"
            pair_label = m1 + " vs " + m2
            print(f"  {pair_label:<40} {r1_str:<14} {r2_str:<14}")

        print()
        print("=" * 60)


# Build two dropdowns — rebuild state list here for safety
compare_state_list = sorted(state_name_to_abbr.keys())

# Set defaults safely
default_1 = "California" if "California" in compare_state_list else compare_state_list[0]
default_2 = "Mississippi" if "Mississippi" in compare_state_list else compare_state_list[1]

dropdown_1 = widgets.Dropdown(
    options=compare_state_list,
    value=default_1,
    description="State 1:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="300px"),
)

dropdown_2 = widgets.Dropdown(
    options=compare_state_list,
    value=default_2,
    description="State 2:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="300px"),
)

compare_output = widgets.interactive_output(
    compare_states, {"state_1": dropdown_1, "state_2": dropdown_2}
)

dropdowns_row = widgets.HBox([dropdown_1, dropdown_2])
display(dropdowns_row, compare_output)
